#Import

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

#Create Flag Parameter

In [0]:
# Check whether the Gold dimension table already exists
if spark.catalog.tableExists("store_catalog.gold.dim_crm_sales_details"):
    incremental_flag = "1"
else:
    incremental_flag = "0"

In [0]:
print(incremental_flag)

1


#Create Dimension crm_sales_details

###Fetch Relative Columns

In [0]:
# Load distinct sales records from the Silver layer
df_scr = spark.sql('''
        SELECT DISTINCT * FROM parquet.`abfss://silver@storenapatdatalake.dfs.core.windows.net/crm_sales_details`
    ''')


In [0]:
df_scr.limit(10).display()

order_number,product_number,customer_id,order_date,ship_date,due_date,sales_amount,quantity,price,record_priority
SO65608,BK-T44U-60,27120,2013-09-02,2013-09-09,2013-09-14,1215.0,1,1215.0,1
SO52027,HL-U509-R,27981,2013-01-31,2013-02-07,2013-02-12,35.0,1,35.0,1
SO57411,BC-M005,12447,2013-05-03,2013-05-10,2013-05-15,10.0,1,10.0,1
SO66241,HL-U509-B,16066,2013-09-12,2013-09-19,2013-09-24,35.0,1,35.0,1
SO48958,BK-R89B-52,23310,2012-07-20,2012-07-27,2012-08-01,2182.0,1,2182.0,1
SO63577,CL-9009,23083,2013-08-04,2013-08-11,2013-08-16,8.0,1,8.0,1
SO64605,VE-C304-M,18649,2013-08-19,2013-08-26,2013-08-31,64.0,1,64.0,1
SO66374,HL-U509-R,17941,2013-09-15,2013-09-22,2013-09-27,35.0,1,35.0,1
SO69993,WB-H098,11175,2013-11-03,2013-11-10,2013-11-15,5.0,1,5.0,1
SO53008,TI-M267,13962,2013-02-20,2013-02-27,2013-03-04,25.0,1,25.0,1


###dim_crm_sales_details - Initial and Incremental

In [0]:
# Read existing Gold records or create an empty DataFrame for the initial load
if spark.catalog.tableExists("store_catalog.gold.dim_crm_sales_details") == True:
    df_sink = spark.sql('''
            SELECT
                crm_sales_details_surrogate_key,
                order_number,
                product_number,
                customer_id,
                order_date,
                ship_date,
                due_date,
                sales_amount,
                quantity,
                price
            FROM
                store_catalog.gold.dim_crm_sales_details
        ''')

else:
    df_sink = spark.sql('''
            SELECT
                1 as crm_sales_details_surrogate_key,
                order_number,
                product_number,
                customer_id,
                order_date,
                ship_date,
                due_date,
                sales_amount,
                quantity,
                price
            FROM
                parquet.`abfss://silver@storenapatdatalake.dfs.core.windows.net/crm_sales_details`
            WHERE
                1 = 0
        ''')


###Filtering new records and old records

In [0]:
# Match records using order number, product number, and customer ID
df_filter = df_scr.join(df_sink, (df_scr["order_number"] == df_sink["order_number"]) &(df_scr["product_number"] == df_sink["product_number"])&(df_scr["customer_id"] == df_sink["customer_id"]), "left")\
        .select(df_scr["order_number"],df_scr["product_number"],df_scr["customer_id"],
            df_scr["order_date"],df_scr["ship_date"],df_scr["due_date"],
            df_scr["sales_amount"],df_scr["quantity"],df_scr["price"],
            df_sink["crm_sales_details_surrogate_key"]
    )

In [0]:
# Check for duplicate surrogate key matches
df_filter.groupBy("crm_sales_details_surrogate_key") \
    .count() \
    .filter(col("count") > 1) \
    .limit(10)\
    .show(truncate=False)

+-------------------------------+-----+
|crm_sales_details_surrogate_key|count|
+-------------------------------+-----+
+-------------------------------+-----+



**df_filter_old**

In [0]:
# Separate existing sales records
df_filter_old = df_filter.filter(col("crm_sales_details_surrogate_key").isNotNull())

**df_filter_new**

In [0]:
# Select new sales records that need surrogate keys
df_filter_new = df_filter.filter(col("crm_sales_details_surrogate_key").isNull())\
    .select(df_scr["order_number"],df_scr["product_number"],df_scr["customer_id"],
            df_scr["order_date"],df_scr["ship_date"],df_scr["due_date"],
            df_scr["sales_amount"],df_scr["quantity"],df_scr["price"]
    )

In [0]:
df_filter_new.limit(10).display()

order_number,product_number,customer_id,order_date,ship_date,due_date,sales_amount,quantity,price


###Create Surrogate Key

**Fetch the max Surrogate Key from existing table**

In [0]:
# Retrieve the next available surrogate key
if (incremental_flag == "0"):
    max_value = 1
else:
    max_value_df = spark.sql('''SELECT MAX(crm_sales_details_surrogate_key) FROM store_catalog.gold.dim_crm_sales_details''')
    max_value = max_value_df.collect()[0][0]+1

**Create Surrogate Key column and Add the max surrogate key**

In [0]:
# Assign surrogate keys to new sales records
df_filter_new = df_filter_new.withColumn("crm_sales_details_surrogate_key",max_value + monotonically_increasing_id())
df_filter_new.limit(10).display()

order_number,product_number,customer_id,order_date,ship_date,due_date,sales_amount,quantity,price,crm_sales_details_surrogate_key


###Create Final DF - df_filter_old + df_filter_new

In [0]:
# Combine existing records with newly identified sales records
df_final = df_filter_new.union(df_filter_old)

In [0]:
df_final.limit(10).display()

order_number,product_number,customer_id,order_date,ship_date,due_date,sales_amount,quantity,price,crm_sales_details_surrogate_key
SO65608,BK-T44U-60,27120,2013-09-02,2013-09-09,2013-09-14,1215.0,1,1215.0,1
SO52027,HL-U509-R,27981,2013-01-31,2013-02-07,2013-02-12,35.0,1,35.0,2
SO57411,BC-M005,12447,2013-05-03,2013-05-10,2013-05-15,10.0,1,10.0,3
SO66241,HL-U509-B,16066,2013-09-12,2013-09-19,2013-09-24,35.0,1,35.0,4
SO48958,BK-R89B-52,23310,2012-07-20,2012-07-27,2012-08-01,2182.0,1,2182.0,5
SO63577,CL-9009,23083,2013-08-04,2013-08-11,2013-08-16,8.0,1,8.0,6
SO64605,VE-C304-M,18649,2013-08-19,2013-08-26,2013-08-31,64.0,1,64.0,7
SO66374,HL-U509-R,17941,2013-09-15,2013-09-22,2013-09-27,35.0,1,35.0,8
SO69993,WB-H098,11175,2013-11-03,2013-11-10,2013-11-15,5.0,1,5.0,9
SO53008,TI-M267,13962,2013-02-20,2013-02-27,2013-03-04,25.0,1,25.0,10


# SCD TYPE - 1 (UPSERT)

In [0]:
from delta.tables import DeltaTable

In [0]:
if spark.catalog.tableExists("store_catalog.gold.dim_crm_sales_details"):
    # Incremental Run
    # Merge sales records into the Gold dimension table
    delta_tbl = DeltaTable.forPath(spark,"abfss://gold@storenapatdatalake.dfs.core.windows.net/dim_crm_sales_details")

    delta_tbl.alias("trg").merge(df_final.alias("src"),"trg.crm_sales_details_surrogate_key = src.crm_sales_details_surrogate_key")\
                            .whenMatchedUpdateAll()\
                            .whenNotMatchedInsertAll()\
                            .execute()

else: 
    # Initial Run
    # Create the Gold dimension table during the initial load
    df_final.write.format("delta")\
        .mode("overwrite")\
        .option("path","abfss://gold@storenapatdatalake.dfs.core.windows.net/dim_crm_sales_details")\
        .saveAsTable("store_catalog.gold.dim_crm_sales_details")


In [0]:
%sql
SELECT * FROM store_catalog.gold.dim_crm_sales_details  ORDER BY crm_sales_details_surrogate_key DESC LIMIT 10

order_number,product_number,customer_id,order_date,ship_date,due_date,sales_amount,quantity,price,crm_sales_details_surrogate_key
SO73247,BK-R79Y-48,21297,2013-12-16,2013-12-23,2013-12-28,1701.0,1,1701.0,8589970960
SO63768,BK-M68S-46,17199,2013-08-07,2013-08-14,2013-08-19,2320.0,1,2320.0,8589970959
SO72319,HL-U509-B,18416,2013-12-04,2013-12-11,2013-12-16,35.0,1,35.0,8589970958
SO63449,WB-H098,12959,2013-08-02,2013-08-09,2013-08-14,5.0,1,5.0,8589970957
SO59487,BK-R19B-48,22581,2013-06-04,2013-06-11,2013-06-16,540.0,1,540.0,8589970956
SO54865,TI-R092,26757,2013-03-22,2013-03-29,2013-04-03,21.0,1,21.0,8589970955
SO48969,BK-R89B-58,23320,2012-07-21,2012-07-28,2012-08-02,2182.0,1,2182.0,8589970954
SO66243,BK-M38S-40,14852,2013-09-12,2013-09-19,2013-09-24,769.0,1,769.0,8589970953
SO65001,WB-H098,12774,2013-08-26,2013-09-02,2013-09-07,5.0,1,5.0,8589970952
SO73264,HY-1023-70,13624,2013-12-17,2013-12-24,2013-12-29,55.0,1,55.0,8589970951


In [0]:
%sql
SELECT COUNT(*) FROM store_catalog.gold.dim_crm_sales_details 

count(1)
60398
